# RQ2: Mental Disorder Association Analysis and ICD Alignment

This notebook builds a Power BI-ready analysis pipeline to identify language-based associations between mental-disorder proxy labels derived from Reddit subreddits and compare those associations with broad ICD categories.


## Verified Sources and Cautions

- WHO ICD overview: https://www.who.int/classifications/classification-of-diseases
- WHO ICD portal: https://icd.who.int/en
- WHO ICD-11 browser: https://icd.who.int/browse11
- scikit-learn `MultinomialNB`: https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html
- scikit-learn `LinearSVC`: https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
- scikit-learn `precision_score`: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html

Reddit subreddits are not diagnoses. All subreddit-to-disorder and disorder-to-ICD links in this notebook are proxy mappings and are explicitly audited for uncertainty.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from __future__ import annotations

import random
import re
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# Path Configuration
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RQ2_DIR = PROJECT_ROOT / "Notebooks" / "RQ2"
OUTPUT_DIR = RQ2_DIR / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
EXPORT_DIR = OUTPUT_DIR / "powerbi"

for path in [RQ2_DIR, OUTPUT_DIR, FIG_DIR, TABLE_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path("/content/drive/MyDrive/reddit_mh_clean.parquet")
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Memory Optimization Utilities
def downcast_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Reduce memory footprint by downcasting numeric types and converting objects to categories."""
    cols = df.dtypes
    for col, dtype in cols.items():
        if "int" in str(dtype):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif "float" in str(dtype):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif dtype == "object" and col != "clean_text":
            df[col] = df[col].astype("category")
    return df

def safe_rate(num: float, den: float) -> float:
    """Return 0 if denominator is 0 to avoid division errors in metrics."""
    return 0.0 if den == 0 else float(num / den)

SOURCE_LOG = pd.DataFrame([
    {"name": "WHO ICD Overview", "url": "https://www.who.int/classifications/classification-of-diseases", "use": "classification context"},
    {"name": "WHO ICD Portal", "url": "https://icd.who.int/en", "use": "ICD version context"},
    {"name": "WHO ICD-11 Browser", "url": "https://icd.who.int/browse11", "use": "broad category naming"},
    {"name": "sklearn MultinomialNB", "url": "https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html", "use": "NB assumptions"},
    {"name": "sklearn LinearSVC", "url": "https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html", "use": "SVM regularization"},
    {"name": "sklearn precision_score", "url": "https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html", "use": "metric definition"},
])
SOURCE_LOG.to_csv(TABLE_DIR / "verified_source_log.csv", index=False)
display(SOURCE_LOG)

,name,url,use
0,WHO ICD Overview,https://www.who.int/classifications/classifica...,classification context
1,WHO ICD Portal,https://icd.who.int/en,ICD version context
2,WHO ICD-11 Browser,https://icd.who.int/browse11,broad category naming
3,sklearn MultinomialNB,https://scikit-learn.org/stable/modules/genera...,NB assumptions
4,sklearn LinearSVC,https://scikit-learn.org/stable/modules/genera...,SVM regularization
5,sklearn precision_score,https://scikit-learn.org/stable/modules/genera...,metric definition


In [ ]:
REQUIRED_COLUMNS = ["clean_text", "subreddit", "timeframe", "is_mental_health", "covid_period"]

def normalize_text(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# Optimizing Load: Read only required columns and use downcasting
table = pq.read_table(str(DATASET_PATH), columns=REQUIRED_COLUMNS)
df_raw = table.to_pandas()
df_raw = downcast_dataframe(df_raw)

# Further cleanup
df_raw["clean_text"] = normalize_text(df_raw["clean_text"])
df_raw["row_id"] = np.arange(len(df_raw), dtype=np.int32)
df_raw["empty_clean_text"] = df_raw["clean_text"].eq("")

dataset_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": int(len(df_raw))},
        {"metric": "columns", "value": int(df_raw.shape[1])},
        {"metric": "subreddits", "value": int(df_raw["subreddit"].nunique())},
        {"metric": "empty_clean_text_rows", "value": int(df_raw["empty_clean_text"].sum())},
        {"metric": "mental_health_share", "value": float(df_raw["is_mental_health"].mean())},
    ]
)
dataset_summary.to_csv(TABLE_DIR / "dataset_summary.csv", index=False)
display(dataset_summary)

,metric,value
0,rows,1.107302e+06
1,columns,7.000000e+00
2,subreddits,2.800000e+01
3,empty_clean_text_rows,1.900000e+01
4,mental_health_share,4.261096e-01


In [ ]:
ICD_VERSION = "ICD-11"

ICD_PROXY_MAP = {
    "adhd": ("attention_deficit_hyperactivity_disorder", "neurodevelopmental_disorders", "high", True, "Direct ADHD name match; still a proxy."),
    "autism": ("autism_spectrum_disorder", "neurodevelopmental_disorders", "high", True, "Direct autism name match; still a proxy."),
    "anxiety": ("anxiety_or_fear_related_disorder", "anxiety_or_fear_related_disorders", "high", True, "Broad anxiety proxy from subreddit name."),
    "healthanxiety": ("health_anxiety_related", "anxiety_or_fear_related_disorders", "medium", True, "Approximate anxiety-related proxy."),
    "socialanxiety": ("social_anxiety_related", "anxiety_or_fear_related_disorders", "high", True, "Strong social-anxiety proxy."),
    "depression": ("depressive_disorder_related", "mood_disorders", "high", True, "Strong depression proxy."),
    "bipolarreddit": ("bipolar_or_related_disorder", "mood_disorders", "high", True, "Strong bipolar proxy."),
    "bpd": ("borderline_pattern_related", "personality_disorders", "medium", True, "BPD shorthand is somewhat ambiguous."),
    "ptsd": ("post_traumatic_stress_related", "disorders_specifically_associated_with_stress", "high", True, "Strong PTSD proxy."),
    "schizophrenia": ("schizophrenia_or_other_primary_psychotic_disorder", "psychotic_disorders", "high", True, "Strong schizophrenia proxy."),
    "EDAnonymous": ("eating_disorder_related", "feeding_or_eating_disorders", "medium", True, "Eating-disorder proxy is plausible but broad."),
    "addiction": ("substance_or_addictive_behaviour_related", "disorders_due_to_substance_use_or_addictive_behaviours", "medium", True, "Broad addiction proxy."),
    "alcoholism": ("alcohol_related_disorder", "disorders_due_to_substance_use_or_addictive_behaviours", "high", True, "Alcohol-related proxy."),
    "suicidewatch": ("suicidality_related_support", "risk_and_crisis_support", "low", False, "Support/crisis forum rather than single disorder."),
    "mentalhealth": ("general_mental_health_discussion", "transdiagnostic_mental_health", "low", False, "General discussion, not disorder-specific."),
    "lonely": ("loneliness_distress_related", "social_distress_non_specific", "low", False, "Distress-related but not clean ICD disorder proxy."),
}

CONTROL_SUBREDDITS = {
    "conspiracy", "COVID19_support", "divorce", "fitness", "guns", "jokes", "legaladvice",
    "meditation", "parenting", "personalfinance", "relationships", "teaching"
}

mapping_rows = []
for subreddit in sorted(df_raw["subreddit"].dropna().astype(str).unique()):
    if subreddit in ICD_PROXY_MAP:
        label, group, confidence, include_flag, note = ICD_PROXY_MAP[subreddit]
        mapping_rows.append({
            "subreddit": subreddit,
            "mapping_status": "mapped_proxy",
            "is_mental_health_proxy": 1,
            "proxy_disorder_label": label,
            "proxy_group_label": group,
            "icd_proxy_category": "Mental, behavioural or neurodevelopmental disorders",
            "confidence": confidence,
            "include_in_main_multiclass": include_flag,
            "uncertainty_reason": note,
            "icd_version": ICD_VERSION,
        })
    elif subreddit in CONTROL_SUBREDDITS:
        mapping_rows.append({
            "subreddit": subreddit,
            "mapping_status": "control_group",
            "is_mental_health_proxy": 0,
            "proxy_disorder_label": "control_non_mental_health",
            "proxy_group_label": "control_group",
            "icd_proxy_category": "not_applicable",
            "confidence": "high",
            "include_in_main_multiclass": False,
            "uncertainty_reason": "Control subreddit for contrast only.",
            "icd_version": ICD_VERSION,
        })
    else:
        mapping_rows.append({
            "subreddit": subreddit,
            "mapping_status": "review_required",
            "is_mental_health_proxy": np.nan,
            "proxy_disorder_label": "review_required",
            "proxy_group_label": "review_required",
            "icd_proxy_category": "review_required",
            "confidence": "low",
            "include_in_main_multiclass": False,
            "uncertainty_reason": "Manual review required before ICD interpretation.",
            "icd_version": ICD_VERSION,
        })

icd_map_df = pd.DataFrame(mapping_rows)
icd_map_df["is_uncertain"] = icd_map_df["confidence"].ne("high")
icd_map_df.to_csv(TABLE_DIR / "icd_proxy_mapping.csv", index=False)
display(icd_map_df)


,subreddit,mapping_status,is_mental_health_proxy,proxy_disorder_label,proxy_group_label,icd_proxy_category,confidence,include_in_main_multiclass,uncertainty_reason,icd_version,is_uncertain
0,COVID19_support,control_group,0,control_non_mental_health,control_group,not_applicable,high,False,Control subreddit for contrast only.,ICD-11,False
1,EDAnonymous,mapped_proxy,1,eating_disorder_related,feeding_or_eating_disorders,"Mental, behavioural or neurodevelopmental diso...",medium,True,Eating-disorder proxy is plausible but broad.,ICD-11,True
2,addiction,mapped_proxy,1,substance_or_addictive_behaviour_related,disorders_due_to_substance_use_or_addictive_be...,"Mental, behavioural or neurodevelopmental diso...",medium,True,Broad addiction proxy.,ICD-11,True
3,adhd,mapped_proxy,1,attention_deficit_hyperactivity_disorder,neurodevelopmental_disorders,"Mental, behavioural or neurodevelopmental diso...",high,True,Direct ADHD name match; still a proxy.,ICD-11,False
4,alcoholism,mapped_proxy,1,alcohol_related_disorder,disorders_due_to_substance_use_or_addictive_be...,"Mental, behavioural or neurodevelopmental diso...",high,True,Alcohol-related proxy.,ICD-11,False
5,anxiety,mapped_proxy,1,anxiety_or_fear_related_disorder,anxiety_or_fear_related_disorders,"Mental, behavioural or neurodevelopmental diso...",high,True,Broad anxiety proxy from subreddit name.,ICD-11,False
6,autism,mapped_proxy,1,autism_spectrum_disorder,neurodevelopmental_disorders,"Mental, behavioural or neurodevelopmental diso...",high,True,Direct autism name match; still a proxy.,ICD-11,False
7,bipolarreddit,mapped_proxy,1,bipolar_or_related_disorder,mood_disorders,"Mental, behavioural or neurodevelopmental diso...",high,True,Strong bipolar proxy.,ICD-11,False
8,bpd,mapped_proxy,1,borderline_pattern_related,personality_disorders,"Mental, behavioural or neurodevelopmental diso...",medium,True,BPD shorthand is somewhat ambiguous.,ICD-11,True
9,conspiracy,control_group,0,control_non_mental_health,control_group,not_applicable,high,False,Control subreddit for contrast only.,ICD-11,False


In [ ]:
df_model = df_raw.merge(icd_map_df, on="subreddit", how="left")
df_model["binary_target"] = df_model["is_mental_health"].astype(int)
df_model["proxy_multiclass_target"] = df_model["proxy_disorder_label"].fillna("review_required")
df_model["is_main_multiclass_eligible"] = (
    df_model["binary_target"].eq(1)
    & df_model["include_in_main_multiclass"].fillna(False)
    & df_model["clean_text"].ne("")
)

cohort_summary = pd.DataFrame(
    [
        {"metric": "binary_rows", "value": int(len(df_model))},
        {"metric": "multiclass_rows", "value": int(df_model["is_main_multiclass_eligible"].sum())},
        {"metric": "multiclass_labels", "value": int(df_model.loc[df_model["is_main_multiclass_eligible"], "proxy_multiclass_target"].nunique())},
        {"metric": "uncertain_mapping_rows", "value": int(df_model["is_uncertain"].sum())},
    ]
)
cohort_summary.to_csv(TABLE_DIR / "cohort_summary.csv", index=False)
display(cohort_summary)

powerbi_contract = pd.DataFrame(
    [
        {"table_name": "dim_icd_mapping", "grain": "one row per subreddit"},
        {"table_name": "dim_sources", "grain": "one row per verified source"},
        {"table_name": "fact_binary_metrics", "grain": "one row per model and split"},
        {"table_name": "fact_multiclass_metrics", "grain": "one row per model, split, and label"},
        {"table_name": "fact_confusion_matrix", "grain": "one row per true-predicted pair"},
        {"table_name": "fact_regularization", "grain": "one row per penalty setting and split"},
        {"table_name": "fact_feature_importance", "grain": "one row per model/class/rank"},
        {"table_name": "fact_disorder_association", "grain": "one row per disorder pair"},
        {"table_name": "fact_mapping_uncertainty", "grain": "one row per subreddit"},
    ]
)
powerbi_contract.to_csv(TABLE_DIR / "powerbi_contract.csv", index=False)
display(powerbi_contract)


,metric,value
0,binary_rows,1107302
1,multiclass_rows,336698
2,multiclass_labels,13
3,uncertain_mapping_rows,190288


,table_name,grain
0,dim_icd_mapping,one row per subreddit
1,dim_sources,one row per verified source
2,fact_binary_metrics,one row per model and split
3,fact_multiclass_metrics,"one row per model, split, and label"
4,fact_confusion_matrix,one row per true-predicted pair
5,fact_regularization,one row per penalty setting and split
6,fact_feature_importance,one row per model/class/rank
7,fact_disorder_association,one row per disorder pair
8,fact_mapping_uncertainty,one row per subreddit


In [ ]:
def add_split_column(frame: pd.DataFrame, target_col: str, split_col: str) -> pd.DataFrame:
    usable = frame.loc[frame["clean_text"].ne("")].copy()
    train_idx, temp_idx = train_test_split(
        usable.index,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=usable[target_col],
    )
    valid_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=usable.loc[temp_idx, target_col],
    )
    result = frame.copy()
    result[split_col] = "unused"
    result.loc[train_idx, split_col] = "train"
    result.loc[valid_idx, split_col] = "valid"
    result.loc[test_idx, split_col] = "test"
    return result

def prepare_xy(frame: pd.DataFrame, split_col: str, split_name: str, target_col: str) -> tuple[pd.Series, pd.Series]:
    subset = frame.loc[frame[split_col].eq(split_name) & frame["clean_text"].ne("")]
    return subset["clean_text"], subset[target_col]

def flatten_confusion(cm: np.ndarray, labels: list[str], model_name: str, split_name: str, task_name: str) -> pd.DataFrame:
    rows = []
    for i, true_label in enumerate(labels):
        for j, pred_label in enumerate(labels):
            rows.append({
                "task_name": task_name,
                "model_name": model_name,
                "split_name": split_name,
                "true_label": true_label,
                "predicted_label": pred_label,
                "count": int(cm[i, j]),
            })
    return pd.DataFrame(rows)

def fit_bundle(train_text: pd.Series, train_y: pd.Series, vectorizer, estimator) -> dict[str, Any]:
    if vectorizer is None:
        x_train = np.zeros((len(train_y), 1))
        fitted_estimator = clone(estimator).fit(x_train, train_y)
        return {"vectorizer": None, "estimator": fitted_estimator}
    vec = clone(vectorizer)
    x_train = vec.fit_transform(train_text)
    fitted_estimator = clone(estimator).fit(x_train, train_y)
    return {"vectorizer": vec, "estimator": fitted_estimator}

def predict_bundle(bundle: dict[str, Any], text: pd.Series) -> np.ndarray:
    if bundle["vectorizer"] is None:
        x = np.zeros((len(text), 1))
    else:
        x = bundle["vectorizer"].transform(text)
    return bundle["estimator"].predict(x)

def per_label_metrics(cm: np.ndarray, labels: list[str], model_name: str, split_name: str, task_name: str) -> pd.DataFrame:
    rows = []
    total = cm.sum()
    for idx, label in enumerate(labels):
        tp = cm[idx, idx]
        fn = cm[idx, :].sum() - tp
        fp = cm[:, idx].sum() - tp
        tn = total - tp - fp - fn
        rows.append({
            "task_name": task_name,
            "model_name": model_name,
            "split_name": split_name,
            "label": label,
            "support": int(cm[idx, :].sum()),
            "precision": safe_rate(tp, tp + fp),
            "recall": safe_rate(tp, tp + fn),
            "fpr": safe_rate(fp, fp + tn),
            "f1": safe_rate(2 * tp, 2 * tp + fp + fn),
        })
    return pd.DataFrame(rows)

VECTORIZERS = {
    "count_uni_bi": CountVectorizer(lowercase=True, ngram_range=(1, 2), min_df=5, max_df=0.95),
    "tfidf_uni_bi": TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=5, max_df=0.95, sublinear_tf=True),
}

df_model = add_split_column(df_model, "binary_target", "binary_split")
multiclass_df = add_split_column(df_model.loc[df_model["is_main_multiclass_eligible"]].copy(), "proxy_multiclass_target", "multiclass_split")


In [ ]:
binary_specs = [
    ("dummy_stratified", None, DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)),
    ("multinomial_nb_count", "count_uni_bi", MultinomialNB(alpha=1.0)),
    ("multinomial_nb_tfidf", "tfidf_uni_bi", MultinomialNB(alpha=1.0)),
    ("linearsvc_l2_tfidf", "tfidf_uni_bi", LinearSVC(C=1.0, penalty="l2", loss="squared_hinge", dual=False, random_state=RANDOM_STATE)),
]

# Optimization: Use a 10% stratified sample to reduce execution time drastically
SAMPLE_FRAC = 0.1
df_sample = df_model.groupby("subreddit", group_keys=False).apply(lambda x: x.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE))
print(f"Using {len(df_sample)} rows for faster execution (Sample Rate: {SAMPLE_FRAC})")

# Optimization: Pre-calculate vectorizations on the sample
vectorized_data = {}
for vec_name, vec_obj in VECTORIZERS.items():
    print(f"Pre-vectorizing with {vec_name}...")
    vectorized_data[vec_name] = vec_obj.fit_transform(df_sample["clean_text"])

binary_metrics_rows = []
binary_confusions = []
binary_labels = [0, 1]

# Get indices for splits relative to the sample dataframe
train_idx = df_sample.index[df_sample["binary_split"] == "train"]
valid_idx = df_sample.index[df_sample["binary_split"] == "valid"]
test_idx = df_sample.index[df_sample["binary_split"] == "test"]

# Map indices to integer positions for sparse matrix indexing
sample_indices = {idx: i for i, idx in enumerate(df_sample.index)}

for model_name, vec_key, estimator in binary_specs:
    print(f"Processing model: {model_name}")

    # Prepare training indices
    train_pos = [sample_indices[idx] for idx in train_idx]
    if vec_key is None:
        X_train = np.zeros((len(train_pos), 1))
    else:
        X_train = vectorized_data[vec_key][train_pos]

    y_train = df_sample.loc[train_idx, "binary_target"]
    clf = clone(estimator).fit(X_train, y_train)

    for split_name, idxs in [("train", train_idx), ("valid", valid_idx), ("test", test_idx)]:
        split_pos = [sample_indices[idx] for idx in idxs]
        if vec_key is None:
            X_split = np.zeros((len(split_pos), 1))
        else:
            X_split = vectorized_data[vec_key][split_pos]

        y_true = df_sample.loc[idxs, "binary_target"]
        y_pred = clf.predict(X_split)

        cm = confusion_matrix(y_true, y_pred, labels=binary_labels)
        tn, fp, fn, tp = cm.ravel()

        binary_metrics_rows.append({
            "model_name": model_name,
            "split_name": split_name,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, average="binary", zero_division=0),
            "recall": recall_score(y_true, y_pred, average="binary", zero_division=0),
            "f1": f1_score(y_true, y_pred, average="binary", zero_division=0),
            "fpr": safe_rate(fp, fp + tn),
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        })
        binary_confusions.append(flatten_confusion(cm, [str(l) for l in binary_labels], model_name, split_name, "binary"))

binary_metrics_df = pd.DataFrame(binary_metrics_rows)
binary_confusion_df = pd.concat(binary_confusions, ignore_index=True)
binary_metrics_df.to_csv(TABLE_DIR / "binary_metrics.csv", index=False)
display(binary_metrics_df.sort_values(["split_name", "f1"], ascending=[True, False]))

Using 110729 rows for faster execution (Sample Rate: 0.1)
Pre-vectorizing with count_uni_bi...
Pre-vectorizing with tfidf_uni_bi...
Processing model: dummy_stratified
Processing model: multinomial_nb_count
Processing model: multinomial_nb_tfidf
Processing model: linearsvc_l2_tfidf


,model_name,split_name,accuracy,precision,recall,f1,fpr,tp,fp,fn,tn
11,linearsvc_l2_tfidf,test,0.942069,0.941822,0.922100,0.931857,0.042893,6605,408,558,9104
8,multinomial_nb_tfidf,test,0.909625,0.878075,0.916934,0.897084,0.095879,6568,912,595,8600
5,multinomial_nb_count,test,0.899550,0.837931,0.949881,0.890401,0.138352,6804,1316,359,8196
2,dummy_stratified,test,0.519340,0.439073,0.428591,0.433769,0.412321,3070,3922,4093,5590
9,linearsvc_l2_tfidf,train,0.998758,0.998450,0.998633,0.998541,0.001149,32862,51,45,44330
6,multinomial_nb_tfidf,train,0.939784,0.914085,0.947640,0.930560,0.066042,31184,2931,1723,41450
3,multinomial_nb_count,train,0.926897,0.873496,0.968578,0.918583,0.104008,31873,4616,1034,39765
0,dummy_stratified,train,0.511399,0.426174,0.425928,0.426051,0.425227,14016,18872,18891,25509
10,linearsvc_l2_tfidf,valid,0.942797,0.938810,0.925478,0.932097,0.044442,6582,429,530,9224
7,multinomial_nb_tfidf,valid,0.907963,0.875725,0.912542,0.893755,0.095411,6490,921,622,8732


In [ ]:
multiclass_specs = [
    ("multiclass_nb_count", VECTORIZERS["count_uni_bi"], MultinomialNB(alpha=1.0)),
    ("multiclass_nb_tfidf", VECTORIZERS["tfidf_uni_bi"], MultinomialNB(alpha=1.0)),
    ("multiclass_svc_l2", VECTORIZERS["tfidf_uni_bi"], LinearSVC(C=1.0, penalty="l2", loss="squared_hinge", dual="auto", random_state=RANDOM_STATE)),
    ("multiclass_svc_l1", VECTORIZERS["tfidf_uni_bi"], LinearSVC(C=1.0, penalty="l1", loss="squared_hinge", dual=False, random_state=RANDOM_STATE)),
]

multi_metrics_rows = []
multi_label_rows = []
multi_confusions = []
multi_feature_rows = []
multiclass_labels = sorted(multiclass_df.loc[multiclass_df["multiclass_split"].eq("train"), "proxy_multiclass_target"].astype(str).unique())

x_train_text, y_train = prepare_xy(multiclass_df, "multiclass_split", "train", "proxy_multiclass_target")
for model_name, vectorizer, estimator in multiclass_specs:
    bundle = fit_bundle(x_train_text, y_train, vectorizer, estimator)
    est = bundle["estimator"]
    vec = bundle["vectorizer"]
    for split_name in ["train", "valid", "test"]:
        x_text, y_true = prepare_xy(multiclass_df, "multiclass_split", split_name, "proxy_multiclass_target")
        y_pred = predict_bundle(bundle, x_text)
        cm = confusion_matrix(y_true, y_pred, labels=multiclass_labels)
        multi_metrics_rows.append({
            "model_name": model_name,
            "split_name": split_name,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        })
        multi_label_rows.append(per_label_metrics(cm, multiclass_labels, model_name, split_name, "multiclass"))
        multi_confusions.append(flatten_confusion(cm, multiclass_labels, model_name, split_name, "multiclass"))
    if vec is not None and hasattr(est, "coef_"):
        feature_names = np.array(vec.get_feature_names_out())
        class_names = list(est.classes_)
        for idx, class_name in enumerate(class_names[: est.coef_.shape[0]]):
            top_idx = np.argsort(est.coef_[idx])[::-1][:25]
            for rank, feat_idx in enumerate(top_idx, start=1):
                multi_feature_rows.append({
                    "model_name": model_name,
                    "class_name": class_name,
                    "rank": rank,
                    "feature": feature_names[feat_idx],
                    "coefficient": float(est.coef_[idx, feat_idx]),
                })

multiclass_metrics_df = pd.DataFrame(multi_metrics_rows)
multiclass_per_label_df = pd.concat(multi_label_rows, ignore_index=True)
multiclass_confusion_df = pd.concat(multi_confusions, ignore_index=True)
feature_importance_df = pd.DataFrame(multi_feature_rows)

multiclass_metrics_df.to_csv(TABLE_DIR / "multiclass_metrics.csv", index=False)
multiclass_per_label_df.to_csv(TABLE_DIR / "multiclass_metrics_per_label.csv", index=False)
multiclass_confusion_df.to_csv(TABLE_DIR / "multiclass_confusion_matrix_long.csv", index=False)
feature_importance_df.to_csv(TABLE_DIR / "feature_importance_long.csv", index=False)
display(multiclass_metrics_df.sort_values(["split_name", "f1_macro"], ascending=[True, False]))


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_27189/2958429468.py", line 21, in <cell line: 0>
    y_pred = predict_bundle(bundle, x_text)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_27189/4234647419.py", line 54, in predict_bundle
    x = bundle["vectorizer"].transform(text)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py", line 2128, in transform
    X = super().transform(raw_documents)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py", line 1421, in transform
    _, X = self._count_vocab(raw_documents, fixed_vocab=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/fea

In [ ]:
regularization_specs = [
    ("svc_l2_c0_5", LinearSVC(C=0.5, penalty="l2", loss="squared_hinge", dual="auto", random_state=RANDOM_STATE)),
    ("svc_l2_c1_0", LinearSVC(C=1.0, penalty="l2", loss="squared_hinge", dual="auto", random_state=RANDOM_STATE)),
    ("svc_l2_c2_0", LinearSVC(C=2.0, penalty="l2", loss="squared_hinge", dual="auto", random_state=RANDOM_STATE)),
    ("svc_l1_c0_5", LinearSVC(C=0.5, penalty="l1", loss="squared_hinge", dual=False, random_state=RANDOM_STATE)),
    ("svc_l1_c1_0", LinearSVC(C=1.0, penalty="l1", loss="squared_hinge", dual=False, random_state=RANDOM_STATE)),
    ("svc_l1_c2_0", LinearSVC(C=2.0, penalty="l1", loss="squared_hinge", dual=False, random_state=RANDOM_STATE)),
]

reg_rows = []
x_train_text, y_train = prepare_xy(multiclass_df, "multiclass_split", "train", "proxy_multiclass_target")
for model_name, estimator in regularization_specs:
    bundle = fit_bundle(x_train_text, y_train, VECTORIZERS["tfidf_uni_bi"], estimator)
    est = bundle["estimator"]
    non_zero_ratio = float(np.count_nonzero(est.coef_) / est.coef_.size)
    for split_name in ["train", "valid", "test"]:
        x_text, y_true = prepare_xy(multiclass_df, "multiclass_split", split_name, "proxy_multiclass_target")
        y_pred = predict_bundle(bundle, x_text)
        reg_rows.append({
            "model_name": model_name,
            "split_name": split_name,
            "penalty": est.penalty,
            "C": float(est.C),
            "accuracy": accuracy_score(y_true, y_pred),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "non_zero_coef_ratio": non_zero_ratio,
        })

regularization_df = pd.DataFrame(reg_rows)
regularization_df.to_csv(TABLE_DIR / "regularization_results.csv", index=False)
display(regularization_df)


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_27189/2639311452.py", line 13, in <cell line: 0>
    bundle = fit_bundle(x_train_text, y_train, VECTORIZERS["tfidf_uni_bi"], estimator)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_27189/4234647419.py", line 46, in fit_bundle
    x_train = vec.fit_transform(train_text)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py", line 2104, in fit_transform
    X = super().fit_transform(raw_documents)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr

In [ ]:
best_model_name = multiclass_metrics_df.loc[multiclass_metrics_df["split_name"].eq("valid")].sort_values("f1_macro", ascending=False).iloc[0]["model_name"]
best_confusion = multiclass_confusion_df.loc[(multiclass_confusion_df["model_name"] == best_model_name) & (multiclass_confusion_df["split_name"] == "test")].copy()
association_edges = best_confusion.loc[best_confusion["true_label"] != best_confusion["predicted_label"]].copy()
association_edges["error_total_from_true"] = association_edges.groupby("true_label")["count"].transform("sum")
association_edges["association_strength_error_share"] = association_edges["count"] / association_edges["error_total_from_true"].replace(0, np.nan)
label_to_group = icd_map_df.set_index("proxy_disorder_label")["proxy_group_label"].to_dict()
association_edges["true_group"] = association_edges["true_label"].map(label_to_group)
association_edges["predicted_group"] = association_edges["predicted_label"].map(label_to_group)
association_edges["cross_group_flag"] = association_edges["true_group"] != association_edges["predicted_group"]
association_edges = association_edges.sort_values("association_strength_error_share", ascending=False)
association_edges.to_csv(TABLE_DIR / "disorder_association_edges.csv", index=False)
association_matrix = best_confusion.pivot(index="true_label", columns="predicted_label", values="count").fillna(0)
association_matrix.to_csv(TABLE_DIR / "disorder_association_matrix.csv")
display(association_edges.head(20))


In [ ]:
export_tables = {
    "dim_icd_mapping.csv": icd_map_df,
    "dim_sources.csv": SOURCE_LOG,
    "fact_binary_metrics.csv": binary_metrics_df,
    "fact_multiclass_metrics.csv": multiclass_per_label_df,
    "fact_confusion_matrix.csv": pd.concat([binary_confusion_df, multiclass_confusion_df], ignore_index=True),
    "fact_regularization.csv": regularization_df,
    "fact_feature_importance.csv": feature_importance_df,
    "fact_disorder_association.csv": association_edges,
    "fact_mapping_uncertainty.csv": icd_map_df.loc[icd_map_df["is_uncertain"]].copy(),
}

manifest_rows = []
for file_name, frame in export_tables.items():
    out_path = EXPORT_DIR / file_name
    frame.to_csv(out_path, index=False)
    manifest_rows.append({"file_name": file_name, "output_path": str(out_path), "row_count": int(len(frame)), "column_count": int(frame.shape[1])})

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(TABLE_DIR / "run_manifest.csv", index=False)
manifest_df.to_csv(EXPORT_DIR / "fact_run_manifest.csv", index=False)
display(manifest_df)


## Notebook Outputs

- Tables: `Notebooks/RQ2/outputs/tables/`
- Power BI imports: `Notebooks/RQ2/outputs/powerbi/`
- Figures: `Notebooks/RQ2/outputs/figures/`

Interpretation should focus on text-pattern separation, classifier confusion, and broad ICD-aligned grouping proximity. Subreddit participation should not be treated as equivalent to diagnosis.
